# Regresión logística con scikit-learn: múltiples variables y pipelines


## Bibliotecas


In [ ]:
# Operaciones matemáticas y estadísticas
import pandas as pd
import numpy as np

In [ ]:
# Visualización
import plotly.express as px
import plotly.graph_objs as go

## Conjunto de datos


Utilizaremos un conjunto de datos de *credit scoring* para predecir la calificación del cliente en base a las variables explicativas.

En este caso será un problema de clasificación binaria.


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/pabanib/dataframes/master/credit_card_completo.csv", index_col=0)

In [ ]:
df.head()


Este DataFrame contiene la información del solicitante. El conjunto de datos contiene las siguientes variables:

- **ID**: el número del cliente.

- **CODE_GENDER**: género: M (Masculino) / F (Femenino).

- **FLAG_OWN_CAR**: propietario vehículo: Y (Si) / N (No).

- **FLAG_OWN_REALTY**:  propietario inmueble: Y (Si) / N (No).

- **CNT_CHILDREN**: cantidad de hijos.

- **AMT_INCOME_TOTAL**: ingreso anual.

- **NAME_INCOME_TYPE**: tipo de ingreso.

- **NAME_EDUCATION_TYPE**: nivel educativo.

- **NAME_FAMILY_STATUS**: estado civil.

- **NAME_HOUSING_TYPE**: vivienda.

- **DAYS_BIRTH**: días hasta el nacimiento contando desde hoy hacia atrás, por ejemplo -1 significa ayer.

- **DAYS_EMPLOYED**: días empleado contando desde hoy hacia atrás. Si es positivo, significa la cantidad de días desempleado.

- **FLAG_MOBIL**: tiene número de celular: 1 (Si) / 0 (No).

- **FLAG_WORK_PHONE**: tiene número de teléfono laboral: 1 (Si) / 0 (No).

- **FLAG_PHONE**: tiene número de teléfono fijo: 1 (Si) / 0 (No).

- **FLAG_EMAIL**: tiene dirección de correo electrónico: 1 (Si) / 0 (No).

- **OCCUPATION_TYPE**:	ocupación.

- **CNT_FAM_MEMBERS**:	tamaño del grupo familiar.

- **STATUS**: calificación crediticia: 1 (Mal pagador) / 0 (Buen pagador).


In [ ]:
df.shape

In [ ]:
df.info()

## Preprocesamiento de los datos con `pandas`


Se renombran las columnas para asignarles nombres más descriptivos y significativos. Se utiliza el método `rename()`.


In [ ]:
df.rename(columns={'CODE_GENDER':'Genero',
                   'FLAG_OWN_CAR':'Auto',
                   'FLAG_OWN_REALTY':'Propiedad',
                   'CNT_CHILDREN':'Hijos',
                   'AMT_INCOME_TOTAL':'Ingreso_anual',
                   'NAME_EDUCATION_TYPE':'Nivel_educativo',
                   'NAME_FAMILY_STATUS':'Estado_civil',
                   'NAME_HOUSING_TYPE':'Vivienda',
                   'FLAG_EMAIL':'Email',
                   'FLAG_MOBIL':'Celular',
                   'DAYS_BIRTH':'Dias_nacimiento',
                   'DAYS_EMPLOYED':'Dias_empleado',
                   'NAME_INCOME_TYPE':'Tipo_trabajo',
                   'FLAG_WORK_PHONE':'Telefono_laboral',
                   'FLAG_PHONE':'Telefono_fijo',
                   'CNT_FAM_MEMBERS':'Tamaño_familia',
                   'OCCUPATION_TYPE':'Ocupacion',
                   'STATUS':'Calificacion'},
          inplace=True)
df.head(2)

Con el método `drop()` de `Pandas` se eliminan las columnas del `DataFrame` que no son útiles. Descartamos `ID`, `Celular` (por tener una sola clase) y `Ocupacion` (por tener una gran cantidad de valores faltantes).


In [ ]:
df.drop(columns = ['ID','Celular','Ocupacion'], inplace=True)

Con el método `apply()` de pandas se puede aplicar una función (`lambda`) a lo largo de una columna del `DataFrame` para crear una nueva columna.


In [ ]:
df['Situacion_laboral'] = df.Dias_empleado.apply(lambda x: 'Desempleado' if x >= 0 else 'Empleado')
df['Años_empleado'] = df.Dias_empleado.apply(lambda x: round(-x/365.25, 2) if x < 0 else 0)
df['Edad'] = df.Dias_nacimiento.apply(lambda x: round(-x/365.25, 2))
df.drop(columns=["Dias_nacimiento","Dias_empleado"], inplace=True)
df.head()

El método `replace()` de pandas se utiliza para reemplazar valores específicos en un `DataFrame` con nuevos valores. En este caso se usa un **diccionario de mapeo**.


In [ ]:
df.replace({'Telefono_laboral':{1:'Y',0:'N'},
            'Telefono_fijo':{1:'Y',0:'N'},
            'Email':{1:'Y',0:'N'}},
           inplace=True)

El método `astype()` de pandas se utiliza para cambiar el tipo de datos de una columna en un `DataFrame`.


In [ ]:
df.Tamaño_familia = df.Tamaño_familia.astype(int)
df.head()

## División del conjunto de datos


Separamos la variable objetivo y las variables predictoras o explicativas.


In [ ]:
X = df.drop(columns='Calificacion')
y = df.Calificacion

La función `make_column_selector` del módulo `compose` de `scikit_learn` permite seleccionar columnas específicas de un conjunto de datos.


In [ ]:
from sklearn.compose import make_column_selector as selector
numerical_columns_selector = selector(dtype_exclude=object)
numerical_columns = numerical_columns_selector(X)
X_numerical = df[numerical_columns]
X_numerical.head()

In [ ]:
categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(X)
X_categorical = df[categorical_columns]
X_categorical.head()

## Codificación de variables categóricas


La clase `OneHotEncoder` de la biblioteca `scikit_learn` se utiliza para transformar variables categóricas en representaciones numéricas binarias que pueden ser utilizadas por algoritmos de aprendizaje automático los cuales requieren datos numéricos.


In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output = False)
data_encoded = encoder.fit_transform(X_categorical)
columns_encoded = encoder.get_feature_names_out(X_categorical.columns)
X_categorical_encoded = pd.DataFrame(data_encoded, columns=columns_encoded)
X_categorical_encoded.head()

## Normalización de variables numéricas


La clase `StandardScaler` de la biblioteca `scikit_learn` en Python se utiliza para estandarizar variables numéricas en un conjunto de datos.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_numerical_scaled = scaler.fit_transform(X_numerical)
X_numerical_scaled = pd.DataFrame(X_numerical_scaled, columns=X_numerical.columns)
X_numerical_scaled.head()

In [ ]:
X_numerical_scaled.describe().round(2)

## Combinación de Dataframes


En `pandas`, el método `concat()` se utiliza para concatenar objetos a lo largo de un eje específico. El parámetro `axis` se utiliza para indicar a lo largo de qué eje se realizará la concatenación.

Cuando se utiliza `axis=1` en el método `concat`, se realiza la concatenación a lo largo del eje de las columnas. Esto significa que los objetos se concatenarán uno al lado del otro, creando un nuevo objeto con un mayor número de columnas.


In [ ]:
X_transformed = pd.concat([X_categorical_encoded, X_numerical_scaled], axis=1)
X_transformed.head()

A continuación, se describe el proceso interno que implica la construcción manual de un pipeline.

Este pipeline se compone de una secuencia de pasos que se realizan con el fin de transformar los datos de manera sistemática y eficiente. En primer lugar, se realiza la selección de columnas numéricas y categóricas, con el propósito de distinguir y tratar adecuadamente los diferentes tipos de variables presentes en el conjunto de datos.

Posteriormente, se lleva a cabo la codificación de las variables categóricas utilizando el método de codificación `one-hot`, el cual permite convertir estas variables en una representación binaria. Este proceso es esencial para asegurar que los algoritmos de aprendizaje automático puedan trabajar correctamente con estas variables.

Por otro lado, se realiza el escalado de las variables numéricas mediante el uso de la técnica de estandarización. Esta técnica se encarga de ajustar las variables numéricas de manera que tengan una media de cero y una desviación estándar de uno, lo cual es beneficioso para mejorar el rendimiento de algunos algoritmos de aprendizaje automático.

Finalmente, se lleva a cabo la concatenación de las columnas generadas por ambas transformaciones, obteniendo así un único `DataFrame` transformado que encapsula los cambios realizados. Este enfoque de construcción manual del pipeline asegura que cada paso se realice de manera explícita y controlada, permitiendo una mayor comprensión del proceso y ajuste personalizado de las transformaciones aplicadas a los datos.


### Esquema del preprocesamiento

| Tipo de variable | Transformación | Salida |
|---|---|---|
| Categórica | `OneHotEncoder` | Columnas indicadoras |
| Numérica | `StandardScaler` | Variables estandarizadas |

Las salidas de ambas transformaciones se combinan en una matriz que recibe el modelo.


## División del conjunto de entrenamiento y prueba


Mediante la función `train_test_split()` del módulo `model_selection` de `scikit_learn` se procede a la división del conjunto de datos en conjuntos separados de entrenamiento y prueba.

Esta técnica es comúnmente utilizada en aprendizaje automático para evaluar el rendimiento del modelo y verificar su capacidad de generalización a través de datos no vistos previamente.


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_transformed,
                                                    y,
                                                    random_state=123)

## Ajuste y evaluación del modelo con `sklearn`


En este ejemplo, se crea una instancia de regresión logística mediante la clase `LogisticRegression()`. Luego, se ajusta el modelo a los datos de entrenamiento utilizando el método `fit()`, donde `X_train` representa las variabres predictoras de entrenamiento e `y_train` es la variable objetivo de entrenamiento correspondiente (`STATUS`). Una vez ajustado el modelo, se pueden realizar predicciones en los datos de prueba utilizando el método `predict()`.


In [ ]:
from sklearn.linear_model import LogisticRegression
model_1 = LogisticRegression(max_iter=500).fit(X_train, y_train)

In [ ]:
prediction = model_1.predict(X_test)
prediction[:10]

In [ ]:
tabla = pd.DataFrame({"Prediccion":prediction,
                      "Real":y_test,
                      })
tabla.head()

La clase `LogisticRegression` también proporciona métodos para la evaluación del modelo, como `score()` para calcular la **accuracy** del modelo en los **datos de prueba**.


In [ ]:
model_1.score(X_test, y_test)

La **exactitud** o **accuracy** del modelo es 0.868.


# Machine Learning Pipeline


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

Se ha utilizado `categorical_preprocessor` como preprocesador o transformador `OneHotEncoder` con la opción `handle_unknown="ignore"`, lo cual permite ignorar las clases desconocidas durante la codificación.

Es decir, si se encuentra una clase desconocida durante la transformación en los datos de prueba, se ignorará esa clase y se asignará una fila de ceros.

Además, se ha utilizado `numerical_preprocessor` como el preprocesador `StandardScaler`.


In [ ]:
categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

La clase `ColumnTransformer` de `scikit_learn` permite aplicar transformaciones específicas a columnas seleccionadas de un conjunto de datos.

`ColumnTransformer` aplica `OneHotEncoder` a las columnas categóricas seleccionadas (`categorical_columns`) y `StandardScaler` a las columnas numéricas seleccionadas (`numerical_columns`).

El `ColumnTransformer` toma una `lista` de `tuplas`, donde cada `tupla` representa una transformación que se aplicará a un subconjunto de columnas.

Cada tupla tiene tres elementos:

1. El primer elemento de la tupla es un `string` que sirve como nombre descriptivo para la transformación (`'one-hot-encoder'` y `'standard_scaler'`).

2. El segundo elemento de la tupla es el preprocesador o transformador que se utiliza para esa transformación en particular (`categorical_preprocessor` y `numerical_preprocessor`).

3. El tercer elemento de la tupla es una `lista` de nombres de columnas en las que deseas aplicar esa transformación (`categorical_columns` y `numerical_columns`).


In [ ]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('one-hot-encoder', categorical_preprocessor, categorical_columns),
    ('standard_scaler', numerical_preprocessor, numerical_columns)])

Por último, se crea un pipeline que incluye `preprocessor` y el modelo `LogisticRegression`, el parámetro `max_iter` indica el número máximo de iteraciones permitidas durante el entrenamiento del modelo de regresión logística.

En otras palabras, `max_iter` limita la cantidad de veces que el algoritmo puede iterar para ajustar el modelo. Cada iteración actualiza los coeficientes del modelo en base a una función de pérdida que busca mejorar la calidad del ajuste.


In [ ]:
model_2 = make_pipeline(preprocessor, LogisticRegression(max_iter=500))
model_2

## División del conjunto de entrenamiento y prueba (*train - test split*)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    random_state=123)

Se ajusta el modelo con los datos de entrenamiento para hacer predicciones con los datos de prueba.


In [ ]:
model_2.fit(X_train, y_train)

In [ ]:
predicciones = model_2.predict(X_test)


In [ ]:
model_2.score(X_test, y_test)

La **exactitud** o **accuracy** del modelo es 0.868.


In [ ]:
coeficientes = pd.DataFrame({
    "Variable": model_2[0].get_feature_names_out(),
    "Coeficiente": model_2[1].coef_[0]
})

coeficientes["Importancia_abs"] = coeficientes["Coeficiente"].abs()
coeficientes = coeficientes.sort_values(
    "Importancia_abs",
    ascending=False)

coeficientes

**

## Metricas de evaluación
Existen otro tipo de métricas para poder evaluar un método de estimación cuando el problema es de predicción, entre ellos los más conocidos son **precision** y **recall**. Para observar esto miremos lo que se conoce como matriz de confusión.

![Matriz de confusión](./Clases_ML/imagenes/matriz_confusion.png)

Entonces la precision o valor predictivo positivo  va medir a todos aquellos a los que predecimos como positivos y es verdad (verdaderos positivos) sobre todos los que predecimos como verdaderos. Más formal $VP/(VP+FP)$.

El recall o sensibilidad va medir a todos los verdaderos positivos sobre todos los que son positivos realmente. Más formal $VP/(VP+FN)$

La especificidad o razon de verdaderos negativos mide todos aquellos que son verdaderos negativos sobre todos los negativos reales. Más formal $VN/(FP+VN)$

Estos indicadores deben estar balanceados entre sí, no se puede intentar mejorar solo uno. Por ejemplo si aumentamos la precisión llevandola a 1, significa que acertamos a todos los que son verdaderamente positivos, pero esto, probablemente traiga aparejada que tenemos muchos falsos positivos también, por lo tanto puede ser indicio de sobre ajuste (*overfitting*). En cambio si mejoramos el recall, procurando no tener falsos negativos, puede dar indicios de sub-ajuste (*underfitting*).

Existe unfa fórmula que los unifica a la precision y al recall para trabajar de manera unificada, estos se conocen como las medidas $F_1$. En este caso la fórmula es la siguiente:
$$
F_1 = \frac{2}{1/prec+1/rec}
$$

Este indicador va de 0 a 1 siendo 1 el mejor resultado.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

predicciones = model_2.predict(X_test)

matriz_confusion = confusion_matrix(y_test, predicciones)
pd.DataFrame(
    matriz_confusion,
    index=["Real 0", "Real 1"],
    columns=["Predic. Buen pagador", "Predic. Mal pagador"]
)

In [ ]:
def calcular_metricas(y_real, y_predicho):
    return pd.DataFrame({
        "Métrica": ["Precision", "Recall", "F1-score"],
        "Valor": [
            precision_score(y_real, y_predicho),
            recall_score(y_real, y_predicho),
            f1_score(y_real, y_predicho)
        ]
    })

calcular_metricas(y_test, predicciones)

In [ ]:
predicciones

La mayoría de implementaciones de `scikit_learn`, permiten predicir probabilidades cuando se trata de problemas de clasificación. Es importante entender cómo se calculan estos valores para interpretarlos y utilizarlos correctamente.

En el ejemplo anterior, al aplicar método `predict()` se devuelve  0 (si es "buen pagador") o 1 (si no lo es) para cada observación del conjunto de prueba. Sin embargo, no se dispone de ningún tipo de información sobre la seguridad con la que el modelo realiza esta asignación. Con `predict_proba()`, en lugar de una clasificación, se obtiene la probabilidad ($p$) con la que el modelo considera que cada observación puede pertenecer a cada una de las clases.

In [ ]:
predicciones2 = model_2.predict_proba(X_test)[:,1] > 0.15


matriz_confusion = confusion_matrix(y_test, predicciones2)
pd.DataFrame(
    matriz_confusion,
    index=["Real 0", "Real 1"],
    columns=["Predic. Buen pagador", "Predic. Mal pagador"]
)

In [ ]:
calcular_metricas(y_test, predicciones2)

In [ ]:
from sklearn.metrics import precision_recall_curve

probabilidades = model_2.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, probabilidades)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall,
    y=precision,
    mode="lines",
    name="Curva Precision-Recall"
))

fig.update_layout(
    title="Curva de Precision-Recall",
    xaxis_title="Recall",
    yaxis_title="Precision",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    template="plotly_white"
)

fig.show()

# Conclusiones


En este notebook:

- Realizamos la construcción manual de un pipeline

- Utilizamos la biblioteca `scikit_learn` para entrenar un modelo de regresión logística.

- Implementamos un pipeline y aplicamos la validación cruzada. Este pipeline incluyó pasos de preprocesamiento como la codificación de variables categóricas mediante `OneHotEncoder` y el escalado de variables numéricas mediante `StandardScaler`.
